# Deploy VSS via the Orchestrator MCP server and OpenClaw UI

This notebook is meant to run **on the GPU host itself** — a [Brev](https://brev.nvidia.com) instance or any other Linux GPU host — and assumes you have already completed the companion notebook **`deploy_nemoclaw.ipynb`** — i.e. the NemoClaw sandbox is up, the VSS policy is applied, the VSS skills and workspace docs are installed, the `vss_orchestrator` MCP is registered, and the OpenClaw UI is reachable.

A few steps are Brev-specific (the generated remote UI link and `BREV_LINK_DOMAIN` autodetection). On other platforms reach the OpenClaw UI over your own networking — use the SSH tunnel shown in section 4.2.

It walks through the rest of the flow for the **Video Search and Summarization (VSS)** blueprint: prepare the host for local NIM-backed VSS profiles, start the host-side VSS Orchestrator MCP server, open the OpenClaw UI and smoke-test it, then deploy and manage VSS from the UI by chatting with the agent.

**Default launchable (Brev):**  
[video-search-and-summarization-blueprint](https://brev.nvidia.com/launchable/deploy/now?launchableID=env-2tYIjRXL4eMCbH9Az8mJC5WPAI4)

On Brev, if you use that launchable the VM is expected to already have the main prerequisites in place. Otherwise — a different launchable, or another platform — use the preflight cells below to confirm what is missing.

**Required prerequisites**

- Run this notebook with **Python 3.11 or newer**. The MCP helper uses Python 3.11 standard-library APIs.
- Complete **`deploy_nemoclaw.ipynb`** first. Set `NEMOCLAW_SANDBOX_NAME` below to the same sandbox you configured there.
- Make sure the intended VSS checkout is the one resolved by `VSS_REPO_DIR`. By default this notebook uses `~/video-search-and-summarization`; set the `VSS_REPO_DIR` environment variable before launching Jupyter if your checkout lives elsewhere.

**What this notebook covers**

- Set required keys and VSS LLM/VLM options.
- Run preflight checks for the agent environment, MCP helper/config, and host prerequisites.
- Prepare the host for local NIM-backed VSS profiles: install/configure NGC CLI, log Docker into `nvcr.io`, and complete host backend prerequisites (MCP requirements + Docker NVIDIA runtime).
- Start the host-side VSS Orchestrator MCP server, open the OpenClaw UI and smoke-test it, then deploy and manage VSS from the UI by asking the agent to use the `vss_orchestrator__*` MCP tools. Teardown of a deployed VSS stack is also done from the UI (the agent invokes `docker_down`).
- *(Optional)* Verify sandbox-to-host reachability through `host.openshell.internal`.

**Security:** prefer `NVIDIA_API_KEY` and `NGC_CLI_API_KEY` from environment variables or your platform's secret store (e.g. Brev secrets). Do **not** commit notebook outputs that contain credentials or live access tokens.


## 1. Settings

Configure the credentials and hardware profile every VSS deployment needs, and point the notebook at the NemoClaw sandbox you created in `deploy_nemoclaw.ipynb`.

<span style="color:red"><strong>Important:</strong> set <code>NGC_CLI_API_KEY</code> below, and <code>NVIDIA_API_KEY</code> if you use NVIDIA-hosted VSS LLM/VLM endpoints. Credentials can also come from the environment or your platform's secret store (e.g. Brev secrets).</span>

### 1.1 Required settings

These apply to every deployment:

- `NGC_CLI_API_KEY` — NVIDIA legacy API key used to pull VSS container artifacts from `nvcr.io`.
- `NVIDIA_API_KEY` — build.nvidia.com key (`nvapi-...`); used by NVIDIA-hosted VSS LLM/VLM endpoints and passed to the MCP server.
- `HARDWARE_PROFILE` — selects the hardware-specific overlay.
- `NEMOCLAW_SANDBOX_NAME` — the sandbox created in `deploy_nemoclaw.ipynb`; the OpenClaw UI you deploy VSS from lives here.

In [ ]:
# ================== Required settings (always set) ==================
NGC_CLI_API_KEY = ""               # NVIDIA Legacy API key — used to pull VSS artifacts from nvcr.io
NVIDIA_API_KEY = ""                # build.nvidia.com key (nvapi-...) — used by NVIDIA-hosted VSS LLM/VLM
HARDWARE_PROFILE = "RTXPRO6000BW"  # DGX-SPARK | RTXPRO6000BW | H100 | L40S | OTHER

# Sandbox created by deploy_nemoclaw.ipynb — the OpenClaw UI you deploy VSS from lives here.
NEMOCLAW_SANDBOX_NAME = "demo"


### 1.2 Advanced settings (defaults — usually leave alone)

VSS LLM/VLM overrides, GPU device ids, the Brev secure-link domain (`BREV_LINK_DOMAIN`, Brev only; ignored off Brev), and MCP server plumbing. `EXTERNAL_IP` falls back to `hostname -I` when blank. Set `LLM_ENDPOINT_URL` / `VLM_ENDPOINT_URL` non-empty only to override the profile defaults with remote endpoints.

In [ ]:
import os
import subprocess
from pathlib import Path


# ================== VSS settings ==================
LLM_NAME = ""
LLM_ENDPOINT_URL = ""
LLM_MODEL_TYPE = ""
LLM_ENABLE_THINKING = ""
OPENAI_API_KEY = ""            # blank => fall back to NVIDIA_API_KEY (typical for integrate.api.nvidia.com)

# Remote VLM — set VLM_ENDPOINT_URL non-empty to force VLM_MODE=remote in generated.env
VLM_NAME = ""
VLM_ENDPOINT_URL = ""
VLM_MODEL_TYPE = ""

LLM_DEVICE_ID = "0"
VLM_DEVICE_ID = "1"
EXTERNAL_IP = ""                    # blank => resolve from `hostname -I`


# ================== Derived (no need to touch) ==================

HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
NGC_CLI_API_KEY = (NGC_CLI_API_KEY or os.environ.get("NGC_CLI_API_KEY", "")).strip()
HARDWARE_PROFILE = (HARDWARE_PROFILE or os.environ.get("HARDWARE_PROFILE", "RTXPRO6000BW")).strip()
EXTERNAL_IP = (EXTERNAL_IP or os.environ.get("EXTERNAL_IP", "")).strip()
if not EXTERNAL_IP:
    try:
        EXTERNAL_IP = subprocess.check_output(["hostname", "-I"], text=True).split()[0]
    except (subprocess.SubprocessError, IndexError):
        EXTERNAL_IP = ""
# baked at onboard by deploy_nemoclaw.ipynb. Empty = autodetect.
BREV_LINK_DOMAIN = os.environ.get("BREV_LINK_DOMAIN", "").strip()
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
ORCHESTRATOR_MCP_VENV_DIR = AGENT_DIR / ".venv"
ORCHESTRATOR_MCP_PYTHON_VERSION = "3.13"
MCP_CONFIG_PATH = DEPLOY_SCRIPTS_DIR / "vss_orchestrator_mcp_config.yml"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"
UV_BIN_DIR = HOME_DIR / ".local" / "bin" / "uv"
DASHBOARD_PORT = int(os.environ.get("NEMOCLAW_DASHBOARD_PORT", "18789"))
LLM_DEVICE_ID = str(LLM_DEVICE_ID or os.environ.get("LLM_DEVICE_ID", "")).strip()
VLM_DEVICE_ID = str(VLM_DEVICE_ID or os.environ.get("VLM_DEVICE_ID", "")).strip()
LLM_NAME = (LLM_NAME or os.environ.get("LLM_NAME", "")).strip()
LLM_ENDPOINT_URL = (LLM_ENDPOINT_URL or os.environ.get("LLM_ENDPOINT_URL", "")).strip()
LLM_MODEL_TYPE = (LLM_MODEL_TYPE or os.environ.get("LLM_MODEL_TYPE", "")).strip()
LLM_ENABLE_THINKING = (LLM_ENABLE_THINKING or os.environ.get("LLM_ENABLE_THINKING", "")).strip()
OPENAI_API_KEY = (OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY", "")).strip()
VLM_NAME = (VLM_NAME or os.environ.get("VLM_NAME", "")).strip()
VLM_ENDPOINT_URL = (VLM_ENDPOINT_URL or os.environ.get("VLM_ENDPOINT_URL", "")).strip()
VLM_MODEL_TYPE = (VLM_MODEL_TYPE or os.environ.get("VLM_MODEL_TYPE", "")).strip()
MCP_HOST = os.environ.get("VSS_ORCHESTRATOR_MCP_HOST", "0.0.0.0").strip()
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9988"))
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"
HOST_INTERNAL_ALIAS = "host.openshell.internal"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", NEMOCLAW_SANDBOX_NAME).strip()
TEST_PORTS = (3000, 8000, 9988, 30888, 5601, 6006, 9200, 8081, 31000, 9901, 38111, 38112)

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("LOG_PATH:", LOG_PATH)
print("HARDWARE_PROFILE:", HARDWARE_PROFILE)
print("EXTERNAL_IP:", EXTERNAL_IP or "(unresolved)")
print("BREV_LINK_DOMAIN:", BREV_LINK_DOMAIN or "(autodetect)")
print("LLM_DEVICE_ID:", LLM_DEVICE_ID or "(profile default)")
print("VLM_DEVICE_ID:", VLM_DEVICE_ID or "(profile default)")
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("MCP_URL:", MCP_URL)
if LLM_ENDPOINT_URL:
    print("LLM_NAME:", LLM_NAME)
    print("LLM_ENDPOINT_URL:", LLM_ENDPOINT_URL)
    print("LLM_MODEL_TYPE:", LLM_MODEL_TYPE)
print("LLM_ENABLE_THINKING:", LLM_ENABLE_THINKING)
if VLM_ENDPOINT_URL:
    print("VLM_NAME:", VLM_NAME)
    print("VLM_ENDPOINT_URL:", VLM_ENDPOINT_URL)
    print("VLM_MODEL_TYPE:", VLM_MODEL_TYPE)
print("NGC_CLI_API_KEY set:", bool(NGC_CLI_API_KEY))
print("NVIDIA_API_KEY set:", bool(NVIDIA_API_KEY))


## 2. Preflight

Run the next cell to confirm the expected keys, files, and commands are present on the host, load the MCP helper module, and (on Brev) resolve the secure-link domain used by the UI/MCP cells later.

> Docker version pinning is handled in `deploy_nemoclaw.ipynb` (it must run before the OpenClaw sandbox comes up). This notebook assumes Docker is already pinned to the tested range.

In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

OrchestratorTool = orchestrator_mcp_helper.OrchestratorTool
build_vss_ui_url = orchestrator_mcp_helper.build_vss_ui_url
poll_compose_op = orchestrator_mcp_helper.poll_compose_op
require_success = orchestrator_mcp_helper.require_success
tool_call = orchestrator_mcp_helper.tool_call
resolve_openshell_gateway_container = orchestrator_mcp_helper.resolve_openshell_gateway_container
detect_brev_link_domain = orchestrator_mcp_helper.detect_brev_link_domain

if not shutil.which("uv") and UV_BIN_DIR.exists():
    os.environ["PATH"] = f"{UV_BIN_DIR.parent}:{os.environ.get('PATH', '')}"

# Resolve the Brev secure-link domain up front — 4.1 and the UI link cell rely on it.
BREV_LINK_DOMAIN_RESOLVED = BREV_LINK_DOMAIN or detect_brev_link_domain()

required_checks = {
    "NGC_CLI_API_KEY set": bool(NGC_CLI_API_KEY),
    "HARDWARE_PROFILE set": bool(HARDWARE_PROFILE),
    "EXTERNAL_IP resolved": bool(EXTERNAL_IP),
    "services/agent/": AGENT_DIR.is_dir(),
    "services/agent/pyproject.toml": (AGENT_DIR / "pyproject.toml").is_file(),
    "vss_orchestrator_mcp_config.yml": MCP_CONFIG_PATH.is_file(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    # host commands
    "docker": shutil.which("docker") is not None,
    "python3": shutil.which("python3") is not None,
    "curl": shutil.which("curl") is not None,
    "uv": shutil.which("uv") is not None,
}

optional_checks = {
    "orchestrator MCP venv": ORCHESTRATOR_MCP_VENV_DIR.is_dir(),
}

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")


## 3. Prepare the host for local NIM-backed VSS profiles

If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, complete the next three pre-steps on the host first:

1. install and configure the NGC CLI,
2. authenticate Docker to `nvcr.io`, and
3. prepare the `services/agent/` Python environment.

### 3.1 Install and configure NGC CLI

This pre-step checks whether `ngc` is already installed, installs it if needed, and writes the local NGC CLI config using `NGC_CLI_API_KEY`.

In [ ]:
import os
import platform
import shutil
import subprocess


def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr}\n{result.stdout}")
    return result.stdout.strip()


ngc_path = shutil.which("ngc")
if ngc_path:
    version = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {version}")
else:
    arch = platform.machine()
    filename = "ngccli_arm64.zip" if arch in ("aarch64", "arm64") else "ngccli_linux.zip"
    ngc_cli_version = "4.13.0"
    url = (
        "https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/"
        f"versions/{ngc_cli_version}/files/{filename}"
    )

    print(f"Installing NGC CLI {ngc_cli_version} ({filename})...")
    run(f"cd /tmp && curl -fL --retry 3 --retry-delay 2 -o ngc_cli.zip '{url}'")

    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(
            f"NGC CLI download failed: /tmp/ngc_cli.zip is only {size} bytes."
        )

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    version = run("ngc --version 2>&1 | head -1")
    print(f"Installed NGC CLI: {version}")


if not NGC_CLI_API_KEY:
    raise RuntimeError(
        "NGC_CLI_API_KEY is not set. Set it in the notebook settings cell or export it "
        "in the environment before running this step."
    )

print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = nvidia
""")

print("NGC CLI configured.")
print(run("ngc config current"))

### 3.2 Docker login to `nvcr.io`

If you plan to deploy local NIM-backed VSS profiles, authenticate Docker to the NVIDIA Container Registry before using the orchestrator deployment tools.

This pre-step runs `docker login nvcr.io` with `NGC_CLI_API_KEY`.

In [ ]:
import subprocess

if not NGC_CLI_API_KEY:
    raise RuntimeError("NGC_CLI_API_KEY is not set. Export it before running this cell.")

login_result = subprocess.run(
    [
        "docker",
        "login",
        "nvcr.io",
        "--username",
        "$oauthtoken",
        "--password",
        NGC_CLI_API_KEY,
    ],
    capture_output=True,
    text=True,
)
if login_result.returncode != 0:
    raise RuntimeError(f"Docker login to nvcr.io failed\n{login_result.stderr}")

print("Docker login to nvcr.io: OK")

### 3.3 Install VSS host backend prerequisites

Two host-side preparations for local NIM-backed VSS profiles and the orchestrator MCP server:

- **MCP requirements** — install `uv`, create the `services/agent/` virtualenv, and sync the Python packages needed for `uv run nat mcp …`.
- **VSS backend deps** — if Docker does not list the `nvidia` runtime (or a compose-style smoke test fails), run `nvidia-ctk runtime configure` and restart Docker, then verify with `runtime: nvidia`.

> Run both subsections below before starting the orchestrator MCP server.

#### 3.3.1 MCP requirements

The orchestrator MCP server runs from `services/agent/` via `uv run nat mcp ...`. This cell installs `libcairo2-dev`, `pkg-config`, and `python3-dev` if any are missing, auto-installs `uv` if missing, creates `services/agent/.venv` when needed, and runs `uv sync --no-dev`.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

REQUIRED_MCP_APT_PACKAGES = ("libcairo2-dev", "pkg-config", "python3-dev")


def apt_package_installed(package: str) -> bool:
    result = subprocess.run(
        ["dpkg-query", "-W", "-f=${Status}", package],
        capture_output=True,
        text=True,
    )
    return result.returncode == 0 and result.stdout.strip() == "install ok installed"


def ensure_apt_packages(packages: tuple[str, ...]) -> None:
    missing = [package for package in packages if not apt_package_installed(package)]
    if not missing:
        print("Apt packages already installed:", ", ".join(packages))
        return

    print("Installing missing apt packages:", ", ".join(missing))
    subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["sudo", "env", "DEBIAN_FRONTEND=noninteractive", "apt-get", "install", "-y", *missing],
        check=True,
    )


def ensure_uv_on_path() -> None:
    uv_bin_dir = Path.home() / ".local" / "bin"
    os.environ["PATH"] = f"{uv_bin_dir}:{os.environ.get('PATH', '')}"
    if shutil.which("uv") is None:
        print("Installing uv ...")
        installer = subprocess.run(
            ["curl", "-LsSf", "https://astral.sh/uv/install.sh"],
            check=True,
            capture_output=True,
            text=True,
        )
        subprocess.run(["sh"], input=installer.stdout, text=True, check=True)
        os.environ["PATH"] = f"{uv_bin_dir}:{os.environ.get('PATH', '')}"
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed and auto-install failed.")


def uv_env_for_agent() -> dict[str, str]:
    env = os.environ.copy()
    # Do not inherit the notebook kernel venv; uv should use services/agent/.venv.
    env.pop("VIRTUAL_ENV", None)
    return env


def run_uv_sync() -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["uv", "sync", "--no-dev"],
        cwd=str(AGENT_DIR),
        env=uv_env_for_agent(),
        check=False,
        capture_output=True,
        text=True,
    )


ensure_apt_packages(REQUIRED_MCP_APT_PACKAGES)
ensure_uv_on_path()
if not ORCHESTRATOR_MCP_VENV_DIR.is_dir():
    print(f"Creating Python {ORCHESTRATOR_MCP_PYTHON_VERSION} venv in {ORCHESTRATOR_MCP_VENV_DIR} ...")
    subprocess.run(
        ["uv", "venv", "--python", ORCHESTRATOR_MCP_PYTHON_VERSION],
        cwd=str(AGENT_DIR),
        check=True,
    )

print("Installing orchestrator MCP dependencies (uv sync --no-dev in services/agent/) ...")
sync_result = run_uv_sync()
if sync_result.returncode != 0:
    message = (
        "uv sync failed while preparing the orchestrator MCP environment."
        f"\nSTDOUT:\n{sync_result.stdout}"
        f"\nSTDERR:\n{sync_result.stderr}"
    )
    raise RuntimeError(message)

agent_env = uv_env_for_agent()
subprocess.run(
    ["uv", "run", "nat", "mcp", "--help"],
    cwd=str(AGENT_DIR),
    env=agent_env,
    check=True,
)
module_check = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import vss_agents.orchestrator.tools; print('orchestrator MCP module OK')",
    ],
    cwd=str(AGENT_DIR),
    env=agent_env,
    check=True,
    capture_output=True,
    text=True,
)
print(module_check.stdout.strip())
print(f"Orchestrator MCP venv ready in {ORCHESTRATOR_MCP_VENV_DIR}")

#### 3.3.2 VSS backend deps

Local NIM-backed VSS profiles need Docker to expose the NVIDIA runtime. This cell checks Docker runtime registration and a compose config using `runtime: nvidia`; if either check fails, it runs `nvidia-ctk runtime configure --runtime=docker`, restarts Docker, and verifies the runtime again.

In [ ]:
import json
import shutil
import subprocess
import time


def docker_runtimes() -> dict:
    result = subprocess.run(
        ["docker", "info", "--format", "{{json .Runtimes}}"],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"docker info failed:\n{result.stderr}\n{result.stdout}")
    return json.loads(result.stdout or "{}")


def docker_has_nvidia_runtime() -> bool:
    return "nvidia" in docker_runtimes()


def compose_accepts_nvidia_runtime() -> bool:
    compose_yaml = '''
services:
  nvidia-runtime-smoke:
    image: busybox:latest
    runtime: nvidia
    command: ["true"]
'''.strip()
    result = subprocess.run(
        ["docker", "compose", "-f", "-", "config"],
        input=compose_yaml,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        print("Docker compose runtime smoke test failed:")
        print(result.stderr or result.stdout)
    return result.returncode == 0


def restart_docker() -> None:
    if shutil.which("systemctl"):
        subprocess.run(["sudo", "systemctl", "restart", "docker"], check=True)
    else:
        subprocess.run(["sudo", "service", "docker", "restart"], check=True)


def wait_for_docker(timeout_s: int = 60) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        result = subprocess.run(["docker", "info"], capture_output=True, text=True, check=False)
        if result.returncode == 0:
            return
        time.sleep(2)
    raise RuntimeError("Docker did not become ready after restart.")


if shutil.which("docker") is None:
    raise RuntimeError("docker is not installed or not on PATH.")

runtime_ok = docker_has_nvidia_runtime()
compose_ok = compose_accepts_nvidia_runtime()
if not runtime_ok or not compose_ok:
    if shutil.which("nvidia-ctk") is None:
        raise RuntimeError(
            "Docker is missing the nvidia runtime and nvidia-ctk is not on PATH. "
            "Install NVIDIA Container Toolkit, then re-run this cell."
        )
    print("Configuring Docker NVIDIA runtime with nvidia-ctk ...")
    subprocess.run(["sudo", "nvidia-ctk", "runtime", "configure", "--runtime=docker"], check=True)
    restart_docker()
    wait_for_docker()

runtimes = docker_runtimes()
if "nvidia" not in runtimes:
    raise RuntimeError(f"Docker still does not list the nvidia runtime. Runtimes: {sorted(runtimes)}")
if not compose_accepts_nvidia_runtime():
    raise RuntimeError("Docker compose still does not accept runtime: nvidia.")

print("Docker NVIDIA runtime: OK")
print("Docker runtimes:", ", ".join(sorted(runtimes)))

## 4. Start the VSS Orchestrator MCP server and deploy a profile

From here on, the agent — not this notebook — drives the VSS deployment. Work through the sub-steps in this order (note that **4.3** is run before **4.2** in the actual UI flow):

1. **4.1** — start the host-side VSS Orchestrator MCP server (a host process listening on port `9988`).
2. **4.3** — open the OpenClaw UI, then run the smoke tests to confirm the agent can chat, sees the VSS skills, and can reach the orchestrator MCP server.
3. **4.2** — once the smoke tests pass, ask the agent in the UI to use the `vss_orchestrator__*` MCP tools to deploy and manage VSS (including teardown via `docker_down`).

**4.2** is reference material (tool list + sample prompts) and has no code cell of its own — all the work happens in the OpenClaw UI chat once **4.1** and **4.3** are done.


### 4.1 Start the VSS Orchestrator MCP server

Run the next cell after the MCP requirements and VSS backend dependency cells to stop any previously recorded MCP server from this notebook session and start a fresh host-side listener on port `9988`.

The cell checks that the prerequisites completed (`uv`, orchestrator MCP venv, Docker, OpenShell, MCP config), then runs `uv run nat mcp serve` in the background and waits for the health check to pass. Logs are written to `.orchestrator-artifacts/vss_orchestrator_mcp.log`.

In [ ]:
import os
import importlib.util
import shutil
import signal
import subprocess
import sys
import time


helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for label, ok in {
    "uv": shutil.which("uv") is not None,
    "orchestrator MCP venv": ORCHESTRATOR_MCP_VENV_DIR.is_dir(),
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "agent dir": AGENT_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
}.items():
    if not ok:
        raise RuntimeError(f"Cannot start the MCP server because {label} is unavailable. Resolve this before proceeding.")


def _wait_for_mcp_health(process: subprocess.Popen, timeout_s: int = 60, interval_s: int = 3) -> None:
    deadline = time.time() + timeout_s
    last_error = "health check did not run"
    while time.time() < deadline:
        return_code = process.poll()
        if return_code is not None:
            raise RuntimeError(f"MCP server exited before becoming healthy with exit code {return_code}: {last_error}")
        try:
            healthy, message = orchestrator_mcp_helper.check_mcp_health(MCP_URL, AGENT_DIR)
        except subprocess.TimeoutExpired:
            healthy, message = False, "health command timed out"
        last_error = message
        if healthy:
            print(f"MCP health check passed: {message}")
            return
        time.sleep(interval_s)

    raise RuntimeError(f"MCP server did not become healthy within {timeout_s}s: {last_error}")


existing_pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if existing_pid:
    try:
        os.kill(existing_pid, signal.SIGTERM)
        print(f"Stopped existing MCP server PID {existing_pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"Recorded MCP server PID {existing_pid} is no longer running")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
env["BREV_LINK_DOMAIN"] = BREV_LINK_DOMAIN_RESOLVED
if NGC_CLI_API_KEY:
    env["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY
if NVIDIA_API_KEY:
    env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if HARDWARE_PROFILE:
    env["HARDWARE_PROFILE"] = HARDWARE_PROFILE
env["EXTERNAL_IP"] = EXTERNAL_IP
if LLM_ENABLE_THINKING:
    env["LLM_ENABLE_THINKING"] = LLM_ENABLE_THINKING
if LLM_DEVICE_ID:
    env["LLM_DEVICE_ID"] = LLM_DEVICE_ID
if VLM_DEVICE_ID:
    env["VLM_DEVICE_ID"] = VLM_DEVICE_ID
if LLM_ENDPOINT_URL or VLM_ENDPOINT_URL:
    env["OPENAI_API_KEY"] = OPENAI_API_KEY or NVIDIA_API_KEY
if LLM_ENDPOINT_URL:
    env["LLM_ENDPOINT_URL"] = LLM_ENDPOINT_URL
    env["LLM_NAME"] = LLM_NAME
    env["LLM_MODEL_TYPE"] = LLM_MODEL_TYPE
if VLM_ENDPOINT_URL:
    env["VLM_NAME"] = VLM_NAME
    env["VLM_ENDPOINT_URL"] = VLM_ENDPOINT_URL
    env["VLM_MODEL_TYPE"] = VLM_MODEL_TYPE
log_handle = LOG_PATH.open("w")
process = subprocess.Popen(
    [
        "uv",
        "run",
        "nat",
        "mcp",
        "serve",
        "--config_file",
        str(MCP_CONFIG_PATH),
        "--host",
        MCP_HOST,
        "--port",
        str(MCP_PORT),
    ],
    cwd=str(AGENT_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
VSS_ORCHESTRATOR_MCP_PID = process.pid
print(f"Started MCP server with PID {VSS_ORCHESTRATOR_MCP_PID}")
_wait_for_mcp_health(process)
print("MCP log:", LOG_PATH)
print("MCP URL:", MCP_URL)

### 4.2 Open the OpenClaw UI and verify it

Now open the **OpenClaw UI** and run a few quick checks to confirm the agent can chat, sees the VSS skills, and can reach the orchestrator MCP server you started in **4.1**. Once these checks pass, return to **4.2** and drive the deployment via the agent.

#### Step 1. Open the OpenClaw UI

Run the next cell to print a fresh **OpenClaw UI** link, then open it in your browser.

<span style="color:red"><strong>Not on Brev?</strong> If you are accessing the UI from a different machine, open an SSH tunnel before opening the OpenClaw web UI from below:<br/><code>ssh -L 18789:127.0.0.1:18789 &lt;user&gt;@&lt;nemoclaw-host&gt;</code><br/></span>

In [ ]:
import os
import subprocess


def read_etc_environment():
    env = {}
    try:
        with open("/etc/environment", encoding="utf-8") as fp:
            for raw in fp:
                line = raw.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                env[key.strip()] = value.strip().strip('"').strip("'")
    except OSError:
        return env
    return env


def fetch_gateway_token():
    result = subprocess.run(
        ["nemoclaw", "sandbox", "gateway", "token", NEMOCLAW_SANDBOX_NAME, "--quiet"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()


gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container; no UI link generated.")

env_id = os.environ.get("BREV_ENV_ID", "").strip() or read_etc_environment().get("BREV_ENV_ID", "").strip()
dashboard_port = os.environ.get("NEMOCLAW_DASHBOARD_PORT", "18789").strip() or "18789"

if env_id:
    link_domain = os.environ.get("BREV_LINK_DOMAIN", "").strip() or detect_brev_link_domain()
    link_prefix = os.environ.get("BREV_LINK_PREFIX", "").strip() or str(DASHBOARD_PORT)
    origin = f"https://{link_prefix}-{env_id}.{link_domain}"
else:
    origin = f"http://127.0.0.1:{dashboard_port}"
    nemoclaw_host = subprocess.run(
        ["hostname", "-I"], capture_output=True, text=True, check=True
    ).stdout.split()[0]
    ssh_user = os.environ.get("USER", "ubuntu")
    RED, RESET = "\033[31m", "\033[0m"
    print(f"{RED}Make sure an SSH tunnel is running on your laptop before opening the OpenClaw UI:{RESET}")
    print(f"{RED}  $ ssh -L {dashboard_port}:localhost:{dashboard_port} {ssh_user}@{nemoclaw_host}{RESET}")

token = fetch_gateway_token()
openclaw_ui_url = f"{origin}/#token={token}" if token else origin
print("OpenClaw UI:", openclaw_ui_url)

#### Step 2. Verify the LLM works in chat

Start a chat with the agent and send a simple prompt such as:

- `hello`
- `what model are you using?`
- `list your available skills`

![OpenClaw UI chat screenshot](./images/OpenClawUIChat.png)

You should get a normal model response back. If the UI opens but chat fails, re-check provider setup and the active policy. 

If you reproduce the failure from a terminal and see an OpenClaw plugin-loader stack trace, make sure you used `nemoclaw <sandbox> exec -- <command>` rather than `docker exec`; `nemoclaw exec` is the documented path and sets up workspace plugins and gateway routing.

#### Step 3. Verify the VSS skills are imported

Open the **Skills** tab in the UI and confirm the **VSS skills** are present.

![OpenClaw UI skills screenshot](./images/OpenClawUISkills.png)

#### Step 4. Verify the agent sees the `vss_orchestrator` MCP tools

Run the prompts below in order. If any step fails, re-check that **4.1** completed successfully and that the MCP server is still running on port 9988.

**Prompt A — show deployment tools:**

> *"Show me the deployment tools."*

The agent should summarize the VSS deployment tools, including the `vss_orchestrator__*` tools registered by the MCP server.

![OpenClaw UI list tools screenshot](./images/OpenClawUIListTools.png)

**Prompt B — query a tool (list profiles):**

> *"List the available VSS deployment profiles."*

The agent should invoke `vss_orchestrator__profiles` and return `base`, `search`, `alerts`, `lvs`.

![OpenClaw UI search profiles screenshot](./images/OpenClawListVSSProfiles.png)

**Prompt C — invoke a deployment:**

> *"Deploy the VSS `alerts` profile in `verification` mode."*

The agent should chain `vss_orchestrator__docker_generate` → `docker_up` → `docker_status` and stream progress back into the chat. It will ask before invoking the build, and surface a `docker_compose_id` you can reference later.

![OpenClaw UI deploy VSS screenshot](./images/OpenClawUIDeploy.png)

### 4.3 Deploy VSS from the OpenClaw UI via MCP tools

With the MCP server running from **4.1** and **4.3** smoke tests passing, the **OpenClaw agent** can drive the VSS deployment for you through chat. Ask the agent to use the orchestrator tools.

#### Available `vss_orchestrator__*` MCP tools

The MCP server exposes nine tools (all prefixed `vss_orchestrator__`):

| Tool | Purpose |
| --- | --- |
| `profiles` | List supported deployment profiles (`base`, `search`, `alerts`, `lvs`). |
| `prereqs` | Run Docker / GPU / NGC prerequisite checks on the host. |
| `docker_generate` | Resolve `.env` + compose YAML artifacts for the chosen profile. |
| `docker_read` | Fetch generated env/yaml by `docker_compose_id`. |
| `docker_up` | `docker compose up -d --build --quiet-pull` for the generated artifacts. |
| `docker_status` | Poll status/logs of the most recent `docker_up` / `docker_down` operation. |
| `docker_list` | List currently running container names. |
| `docker_logs` | Fetch docker logs for a given container name. |
| `docker_down` | `docker compose down -v --remove-orphans` to tear the deployment back down. |

#### Sample prompts to trigger them

You do **not** call these tools by name — the OpenClaw agent picks the right tool from your natural-language request and chains them in the correct order. Try prompts like:

| Sample prompt | Tool(s) invoked |
| --- | --- |
| *"List the available VSS deployment profiles."* | `profiles` |
| *"Check that my host meets the prerequisites for the `alerts` profile."* | `prereqs` |
| *"Deploy the VSS `alerts` profile in `verification` mode."* | `docker_generate` → `docker_up` → `docker_status` |
| *"Show me the status of the deployment."* | `docker_status` |
| *"List the running VSS containers."* | `docker_list` |
| *"Fetch the last 200 lines of logs from `vss-alert-bridge`."* | `docker_logs` |
| *"Tear down the VSS deployment."* | `docker_down` |

The agent will ask before destructive steps (e.g. `docker_down`) and stream progress back into the chat. If a tool call fails, paste the error message back to the agent and ask it to remediate — it has access to `docker_logs` and `docker_status` to diagnose.

## 5. [OPTIONAL] Verify host reachability from inside the sandbox

<span style="color:red"><strong>Important:</strong> make sure VSS is already deployed on the host before running this step.</span>

Run the next cell only after the agent has finished deploying VSS in **section 4**. It runs `nemoclaw <sandbox> connect` and feeds a probe script into that real sandbox session, which matches the OpenClaw/NemoClaw path used by skills and tools. Do **not** replace this with `docker exec`; that can use a different path and produce misleading results.

The `STATUS` column is intentionally simple:

- `REACHABLE` — an HTTP service answered on that port. `404` and non-policy `403` still count because they prove a service is listening, even if `GET /` is not a valid or authorized route.
- `NOT_REACHABLE` — the port is blocked by policy, no service is listening, DNS failed, or the request timed out. Check the `NOTE` column for the reason.


In [ ]:
import re
import subprocess

required_vars = ("NEMOCLAW_SANDBOX_NAME", "HOST_INTERNAL_ALIAS", "TEST_PORTS")
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError("Required variables are missing: " + ", ".join(missing_vars))

ports = " ".join(str(port) for port in TEST_PORTS)
probe = f"""
HOST_IP="${{HOST_IP:-{HOST_INTERNAL_ALIAS}}}"
PORTS="{ports}"

echo __VSS_PORT_PROBE_BEGIN__
for port in $PORTS; do
  body="$(mktemp)"
  http_code="$(curl -sS --max-time 5 -o "$body" -w '%{{http_code}}' "http://${{HOST_IP}}:${{port}}/" 2>/tmp/portcheck.err)"
  curl_exit=$?
  body_text="$(tr '[:upper:]' '[:lower:]' < "$body")"

  if printf '%s' "$body_text" | grep -Eq 'policy_denied|egress.*denied|not allowed by policy'; then
    status="NOT_REACHABLE"
    note="blocked by policy"
  elif [ "$http_code" = "502" ] || printf '%s' "$body_text" | grep -q 'upstream_unreachable'; then
    status="NOT_REACHABLE"
    note="allowed, no service listening"
  elif [ "$http_code" != "000" ]; then
    status="REACHABLE"
    note="HTTP $http_code response"
  else
    status="NOT_REACHABLE"
    case "$curl_exit" in
      7) note="connect failed" ;;
      28) note="timeout" ;;
      *) note="curl_exit=$curl_exit" ;;
    esac
  fi

  printf '%s|%s|%s|%s\n' "$port" "$status" "$http_code" "$note"
  rm -f "$body" /tmp/portcheck.err
done
echo __VSS_PORT_PROBE_END__
exit
""".strip() + "\n"

print(f"Running reachability probe through: nemoclaw {NEMOCLAW_SANDBOX_NAME} connect")
result = subprocess.run(
    ["nemoclaw", NEMOCLAW_SANDBOX_NAME, "connect"],
    input=probe,
    capture_output=True,
    text=True,
    timeout=120,
)

lines = result.stdout.splitlines()
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
try:
    begin = next(i for i, line in enumerate(lines) if line.strip() == "__VSS_PORT_PROBE_BEGIN__")
    end = next(i for i, line in enumerate(lines[begin + 1 :], start=begin + 1) if line.strip() == "__VSS_PORT_PROBE_END__")
    rows = []
    for line in lines[begin + 1 : end]:
        parts = ansi_escape.sub("", line).strip().split("|", 3)
        if len(parts) == 4 and parts[0].isdigit():
            rows.append(parts)

    print(f"{'PORT':>5}  {'STATUS':<13}  {'HTTP':>4}  NOTE")
    print(f"{'----':>5}  {'------':<13}  {'----':>4}  ----")
    for port, status, http_code, note in rows:
        print(f"{port:>5}  {status:<13}  {http_code:>4}  {note}")
except StopIteration:
    print(result.stdout)

if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"nemoclaw {NEMOCLAW_SANDBOX_NAME} connect probe failed with exit code {result.returncode}"
    )
